# Part 1 — Data Cleaning
Cleans the raw Flickr Tel Aviv dataset before analysis.

**Cleaning steps applied:**
1. Drop rows with any null values
2. Filter years: 2004 < year < 2026
3. Validate month (1–12) and day (1–31)
4. Filter coordinates to the Tel Aviv bounding box
5. Keep only points that fall on land (drop sea/noise)
6. Remove duplicate photo entries (same uid + url)
7. Remove duplicate positions per user per timestamp (GPS jitter)
8. Validate Flickr URL format

Output: `flickr_clean.csv`

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import re

## 1. Load raw data

In [ ]:
RAW_FILE = "flickr_output_TelAviv100.xlsx - flickr_output_100.csv"

df = pd.read_csv(RAW_FILE, skipinitialspace=True)
print(f"Raw rows: {len(df):,}")
df.head()

In [ ]:
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())

## 2. Drop nulls

In [ ]:
before = len(df)
df = df.dropna()
print(f"Dropped {before - len(df):,} rows with nulls — {len(df):,} remain")

## 3. Date validation — year, month, day

In [ ]:
before = len(df)

# Flickr founded 2004; current year is 2026
df = df[(df["year"] > 2004) & (df["year"] < 2026)]

# Sane month and day ranges
df = df[(df["month"] >= 1) & (df["month"] <= 12)]
df = df[(df["day"] >= 1) & (df["day"] <= 31)]

# Build a proper datetime and drop rows that don't parse (e.g. Feb 30)
df["date"] = pd.to_datetime(
    df[["year", "month", "day"]].rename(columns={"year": "year", "month": "month", "day": "day"}),
    errors="coerce"
)
df = df.dropna(subset=["date"])

print(f"Dropped {before - len(df):,} rows with invalid dates — {len(df):,} remain")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")

## 4. Bounding box — Tel Aviv area
Approximate bounds that cover Tel Aviv and its immediate surroundings.

In [ ]:
# Tel Aviv bounding box
LON_MIN, LON_MAX = 34.70, 34.95
LAT_MIN, LAT_MAX = 31.95, 32.20

before = len(df)
df = df[
    (df["lon"] >= LON_MIN) & (df["lon"] <= LON_MAX) &
    (df["lat"] >= LAT_MIN) & (df["lat"] <= LAT_MAX)
]
print(f"Dropped {before - len(df):,} out-of-bounds rows — {len(df):,} remain")

## 5. Land check — drop points in the sea
Uses the Natural Earth land polygons bundled with `geodatasets`.

In [ ]:
import geodatasets

world = gpd.read_file(geodatasets.get_path("naturalearth.land"))
# Work in WGS-84
world = world.to_crs(epsg=4326)

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

before = len(gdf)
# Spatial join — keep only points that intersect a land polygon
gdf = gpd.sjoin(gdf, world[["geometry"]], how="inner", predicate="within")
gdf = gdf.drop(columns=["index_right"])

print(f"Dropped {before - len(gdf):,} sea/water points — {len(gdf):,} remain")

## 6. Remove duplicate photos

In [ ]:
before = len(gdf)
# Same photo uploaded twice (same uid + url)
gdf = gdf.drop_duplicates(subset=["uid", "flickr_url"])
print(f"Dropped {before - len(gdf):,} duplicate photo rows — {len(gdf):,} remain")

## 7. Remove GPS jitter — identical position + timestamp per user

In [ ]:
before = len(gdf)
# Multiple photos from the exact same spot at the exact same time → keep one
gdf = gdf.drop_duplicates(subset=["uid", "lon", "lat", "created_at"])
print(f"Dropped {before - len(gdf):,} GPS-jitter duplicates — {len(gdf):,} remain")

## 8. Validate Flickr URL format

In [ ]:
FLICKR_URL_RE = re.compile(r"^https?://www\.flickr\.com/photos/[^/]+/\d+/?$")

before = len(gdf)
gdf = gdf[gdf["flickr_url"].astype(str).str.match(FLICKR_URL_RE)]
print(f"Dropped {before - len(gdf):,} malformed URL rows — {len(gdf):,} remain")

## 9. Summary

In [ ]:
raw_count = len(pd.read_csv(RAW_FILE, skipinitialspace=True))
print(f"Raw rows:     {raw_count:,}")
print(f"Clean rows:   {len(gdf):,}")
print(f"Removed:      {raw_count - len(gdf):,} ({(raw_count - len(gdf)) / raw_count * 100:.1f}%)")
print(f"Unique users: {gdf['uid'].nunique():,}")
print(f"Date range:   {gdf['date'].min().date()} → {gdf['date'].max().date()}")
gdf.drop(columns=["geometry", "date"]).describe(include="all")

## 10. Save cleaned data

In [ ]:
out = gdf.drop(columns=["geometry", "date"])
out.to_csv("flickr_clean.csv", index=False)
print("Saved → flickr_clean.csv")